# 04 — 贝叶斯优化与主动学习：随堂练习参考答案（Solutions）

本 Notebook 汇总了第四讲 `04a_gaussian_processes.ipynb` 与 `04b_bayesian_optimization.ipynb` 中全部 6 道随堂练习（`04a-1` 至 `04b-3`）的完整可运行参考代码与物理化学解析。

---

## 0. 公共环境与数据读取辅助函数

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from sklearn.decomposition import PCA
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110

def find_file(name):
    for path in (f"../data/{name}", f"data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

# 04a — 高斯过程回归与不确定性量化（Gaussian Processes）练习答案

### 练习 04a-1：补充实验点如何压缩后验不确定性？

**题目回顾**：在手写 `gp_posterior` 函数中，将训练点从 6 个扩充到 12 个（在空白区 `[3.5, 7.2]` 及边界补测 6 个点），对比整个区间上的平均预测标准差 `std_s.mean()` 的变化并并排画图。

In [ ]:
X_train = np.array([1.0, 2.2, 3.5, 7.2, 8.5, 9.3]).reshape(-1, 1)
X_grid  = np.linspace(0, 10, 100).reshape(-1, 1)

def true_mobility(x):
    return np.sin(0.9 * x) + 0.3 * np.cos(2.1 * x)

def rbf_kernel(X1, X2, length_scale=1.2, sigma_f=1.0):
    sq_dist = np.sum(X1**2, axis=1, keepdims=True) + np.sum(X2**2, axis=1) - 2 * X1 @ X2.T
    return (sigma_f ** 2) * np.exp(-0.5 * sq_dist / (length_scale ** 2))

def gp_posterior(X_tr, y_tr, X_te, length_scale=1.2, sigma_f=1.0, noise=1e-4):
    K    = rbf_kernel(X_tr, X_tr, length_scale, sigma_f) + noise * np.eye(len(X_tr))
    K_s  = rbf_kernel(X_tr, X_te, length_scale, sigma_f)
    K_ss = rbf_kernel(X_te, X_te, length_scale, sigma_f) + 1e-8 * np.eye(len(X_te))
    K_inv = np.linalg.inv(K)
    mu_s  = K_s.T @ K_inv @ y_tr
    cov_s = K_ss - K_s.T @ K_inv @ K_s
    return mu_s, np.sqrt(np.maximum(np.diag(cov_s), 0))

X_tr_6  = X_train
X_tr_12 = np.sort(np.vstack([X_train, np.array([[0.3], [4.2], [5.0], [5.8], [6.5], [9.8]])]), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, X_tr, label in zip(axes, [X_tr_6, X_tr_12], ["n = 6", "n = 12"]):
    y_tr = true_mobility(X_tr).ravel()
    mu_s, std_s = gp_posterior(X_tr, y_tr, X_grid)
    print(f"{label:6s} | 全区间平均预测标准差 mean(std) = {std_s.mean():.4f}")
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True Curve")
    ax.plot(X_grid, mu_s, "b-", lw=2, label="GP Mean")
    ax.fill_between(X_grid.ravel(), mu_s - 1.96 * std_s, mu_s + 1.96 * std_s, color="blue", alpha=0.18, label="95% CI")
    ax.scatter(X_tr, y_tr, c="red", s=45, zorder=5, label="Measurements")
    ax.set_title(f"{label} (mean std = {std_s.mean():.3f})")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**结果解析**：在空白区间补测 6 个点后，全区间的平均预测标准差从 `0.264` 骤降至 `0.025`（下降了超过 90%），原本在 $[3.5, 7.2]$ 鼓起的大肚腩置信带完全收拢，后验均值曲线也与真实未知曲线几乎重合。

### 练习 04a-2：对比不同核函数（Matérn vs RBF）的光滑度先验

**题目回顾**：在带噪数据 `(X_train, y_noisy)` 上对比 `Matern(nu=0.5)`、`Matern(nu=1.5)` 和 `RBF()` 的拟合效果。

In [ ]:
rng = np.random.RandomState(7)
y_noisy = true_mobility(X_train).ravel() + rng.normal(0, 0.15, size=len(X_train))

kernels = {
    "Matern (nu=0.5, non-differentiable)": C(1.0) * Matern(1.0, nu=0.5) + WhiteKernel(0.1),
    "Matern (nu=1.5, once differentiable)": C(1.0) * Matern(1.0, nu=1.5) + WhiteKernel(0.1),
    "RBF (infinitely differentiable)":      C(1.0) * RBF(1.0) + WhiteKernel(0.1),
}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharey=True)
for ax, (name, kern) in zip(axes, kernels.items()):
    gp = GaussianProcessRegressor(kernel=kern, n_restarts_optimizer=10, random_state=0)
    gp.fit(X_train, y_noisy)
    mu, std = gp.predict(X_grid, return_std=True)
    ax.plot(X_grid, true_mobility(X_grid), "k--", alpha=0.5, label="True Curve")
    ax.plot(X_grid, mu, "b-", lw=2, label="Mean")
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15, label="95% CI")
    ax.scatter(X_train, y_noisy, c="red", s=50, zorder=5)
    ax.set_title(name, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**结果解析**：
- `Matern(nu=0.5)` 假设底层过程像布朗运动一样粗糙，因此在观测点处呈现尖锐折角，且稍微离开观测点不确定性就急剧张开；
- `Matern(nu=1.5)` 和 `RBF` 都能给出光滑的过渡曲线。在实际材料与化学实验优化中，由于响应面偶有局部陡变（不如无穷阶可导的解析函数那么极端光滑），**`Matern(nu=1.5)` 或 `Matern(nu=2.5)` 往往比 `RBF` 更稳健**。

### 练习 04a-3：预测有机半导体分子的溶解度与不确定性 $\sigma$

**题目回顾**：利用 400 个 AqSolDB 分子训练的 `Tanimoto-GPR`，计算 5 个有机半导体相关分子的预测溶解度、预测不确定性以及与训练集的最大 Tanimoto 相似度。

In [ ]:
df = pd.read_csv(find_file("solubility.csv"))
sol_col = "Solubility" if "Solubility" in df.columns else "logS"

sub = df.sample(700, random_state=42).reset_index(drop=True)
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fps, ys = [], []
for smi, logs in zip(sub["SMILES"], sub[sol_col]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fps.append(fp_gen.GetFingerprintAsNumPy(mol).astype(float))
        ys.append(logs)
    if len(fps) == 400:
        break
X_tr_fp = np.array(fps)
y_tr_fp = np.array(ys)

def tanimoto_kernel(A, B):
    dot = A @ B.T
    na = np.sum(A**2, axis=1, keepdims=True)
    nb = np.sum(B**2, axis=1, keepdims=True).T
    denom = na + nb - dot
    return np.where(denom > 0, dot / denom, 0.0)

K_tr = tanimoto_kernel(X_tr_fp, X_tr_fp) + 0.15 * np.eye(len(X_tr_fp))
K_inv = np.linalg.inv(K_tr)
y_mean = y_tr_fp.mean()

osc_mols = {
    "Benzene":    "c1ccccc1",
    "Thiophene":  "c1ccsc1",
    "Anthracene": "c1ccc2cc3ccccc3cc2c1",
    "Tetracene":  "c1ccc2cc3cc4ccccc4cc3cc2c1",
    "C8-BTBT":    "CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc21",
}
X_osc = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)).astype(float) for s in osc_mols.values()])
K_s = tanimoto_kernel(X_tr_fp, X_osc)

mu_osc  = y_mean + K_s.T @ K_inv @ (y_tr_fp - y_mean)
std_osc = np.sqrt(np.maximum(1.0 - np.sum((K_s.T @ K_inv) * K_s.T, axis=1), 0))
max_sim = K_s.max(axis=0)

pd.DataFrame({
    "Molecule": list(osc_mols.keys()),
    "Pred logS": np.round(mu_osc, 2),
    "Pred std": np.round(std_osc, 3),
    "Max Tanimoto Sim in Train": np.round(max_sim, 3),
})

**结果解析**：
- 苯（`Benzene`）、蒽（`Anthracene`）和并四苯（`Tetracene`）在 400 个训练分子中能找到高度相似的多环芳烃近亲（`Max Tanimoto Sim` 达 `0.60 ~ 1.00`），因此预测标准差 `Pred std` 显著较小（约 `0.34 ~ 0.65`）；
- 相反，`C8-BTBT` 在训练集中既没有 BTBT 硫杂稠环母核、又带着两条长辛基链，与训练集的最大相似度只有 `0.167`，因此 GPR 给出的不确定性高达 `0.946`（几乎接近先验标准差上限 `1.0`），明确提示该分子处于适用域之外。

---

# 04b — 贝叶斯优化与主动实验设计（Bayesian Optimization）练习答案

### 练习 04b-1：调节 UCB 探索权重 $\kappa$ 对跳出局部最优的影响

**题目回顾**：在一维双峰工艺曲线上对比 `kappa = 0.1, 2.0, 10.0` 运行 6 步后的寻优结果。

In [ ]:
def hidden_process(x):
    return 0.6 * np.exp(-0.5 * ((x - 2.3) / 0.8)**2) + 1.0 * np.exp(-0.5 * ((x - 7.8) / 0.9)**2)

X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
X_init = np.array([[1.2], [2.2], [3.6]])
y_init = hidden_process(X_init).ravel()

for kappa in [0.1, 2.0, 10.0]:
    X_obs, y_obs = X_init.copy(), y_init.copy()
    for step in range(1, 7):
        gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
        gp.fit(X_obs, y_obs)
        mu, std = gp.predict(X_grid, return_std=True)
        next_x = X_grid[np.argmax(mu + kappa * std)]
        next_y = hidden_process(next_x).item()
        X_obs = np.vstack([X_obs, next_x.reshape(1, 1)])
        y_obs = np.append(y_obs, next_y)
    print(f"kappa = {kappa:>4} | 6 步后找到的最大值 = {y_obs.max():.4f} (全局最优 = 1.000) | 6 步采样点 = {np.round(X_obs[3:].ravel(), 2)}")

**结果解析**：
- `kappa = 0.1` 过度偏向利用（Exploitation），6 次实验全部围着左侧已知的局部小峰 `x = 2.31` 打转，最终只找到 `0.600`；
- `kappa = 2.0` 和 `kappa = 10.0` 赋予了不确定性足够的权重，第 1 步或第 2 步就果断跨入右侧未知空白区，成功锁定 `x = 7.8` 的全局主峰（`1.000`）。注意 `kappa = 10.0` 由于探索权重过大，还会去测 `x = 10.0` 边界点，而 `kappa = 2.0` 在找到主峰后能更集中地在主峰顶点附近精细收敛。

### 练习 04b-2：二维材料工艺窗口寻优（退火温度 $T$ + 前驱体浓度 $c$）

**题目回顾**：在 $25 \times 25 = 625$ 的二维网格上从 3 个随机初始点出发运行 15 步 EI 贝叶斯优化，对比随机盲抽 18 次的结果并绘制轨迹图。

In [ ]:
def process_2d(X):
    t, c = X[:, 0], X[:, 1]
    peak1 = 0.65 * np.exp(-0.3 * ((t - 2.5)**2 + (c - 3.0)**2))
    peak2 = 1.00 * np.exp(-0.35 * ((t - 7.5)**2 + (c - 7.2)**2))
    return peak1 + peak2

def acq_ei(mu, std, y_best, xi=0.01):
    with np.errstate(divide="ignore", invalid="ignore"):
        imp = mu - y_best - xi
        Z = np.where(std > 1e-9, imp / std, 0.0)
        return np.where(std > 1e-9, imp * norm.cdf(Z) + std * norm.pdf(Z), 0.0)

g = np.linspace(0, 10, 25)
TT, CC = np.meshgrid(g, g)
X_cand = np.column_stack([TT.ravel(), CC.ravel()])
y_true_all = process_2d(X_cand)

rng = np.random.RandomState(42)
init_idx = rng.choice(len(X_cand), size=3, replace=False)
obs_idx = list(init_idx)

for _ in range(15):
    gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.5), alpha=1e-5, random_state=0)
    gp.fit(X_cand[obs_idx], y_true_all[obs_idx])
    unseen = [i for i in range(len(X_cand)) if i not in obs_idx]
    mu, std = gp.predict(X_cand[unseen], return_std=True)
    ei = acq_ei(mu, std, y_true_all[obs_idx].max())
    obs_idx.append(unseen[np.argmax(ei)])

rand_idx = rng.choice(len(X_cand), size=18, replace=False)
print(f"总计 18 次实验: 二维 BO 找到的最大值 = {y_true_all[obs_idx].max():.4f} | 随机盲抽 18 次最大值 = {y_true_all[rand_idx].max():.4f}")

plt.figure(figsize=(6.2, 5))
plt.contourf(TT, CC, y_true_all.reshape(25, 25), levels=20, cmap="viridis")
plt.colorbar(label="Device Performance")
plt.scatter(X_cand[obs_idx[:3], 0], X_cand[obs_idx[:3], 1], c="white", edgecolors="black", s=65, label="Initial (n = 3)")
plt.scatter(X_cand[obs_idx[3:], 0], X_cand[obs_idx[3:], 1], c="red", edgecolors="white", s=80, marker="*", label="BO Selected (n = 15)")
plt.xlabel("Temperature T")
plt.ylabel("Concentration c")
plt.title("2D Bayesian Optimization Trajectory")
plt.legend()
plt.tight_layout()
plt.show()

### 练习 04b-3：在 Buchwald–Hartwig 偶联反应上对比分子指纹（Morgan FP + PCA）与 One-Hot 编码

**题目回顾**：对反应中的 4 个组分计算 256 位 Morgan 指纹并拼接，经 `PCA(30)` 降维后运行贝叶斯优化，与 One-Hot 编码对比。

In [ ]:
bh = pd.read_csv(find_file("bh-reactions.csv"))
y_yield = bh["yield"].values
fp_gen_256 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)

def col_to_fp(series):
    uniq = series.unique()
    cache = {s: fp_gen_256.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)).astype(float) for s in uniq}
    return np.vstack([cache[s] for s in series])

X_fp_raw = np.hstack([col_to_fp(bh[c]) for c in ["ligand", "additive", "base", "aryl halide"]])
X_fp_pca = PCA(n_components=30, random_state=0).fit_transform(X_fp_raw)
X_oh     = pd.get_dummies(bh[["ligand", "additive", "base", "aryl halide"]], dtype=float).values

def run_bo(X, y, seeds=(0, 1, 2), n_init=5, n_steps=20):
    all_curves = []
    for s in seeds:
        rng = np.random.RandomState(s)
        obs = list(rng.choice(len(y), size=n_init, replace=False))
        hist = [y[obs].max()]
        for _ in range(n_steps):
            gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(2.0) + WhiteKernel(1.0), normalize_y=True, random_state=s)
            gp.fit(X[obs], y[obs])
            unseen = np.array([i for i in range(len(y)) if i not in obs])
            mu, std = gp.predict(X[unseen], return_std=True)
            scores = acq_ei(mu, std, y[obs].max(), xi=0.5)
            obs.append(unseen[np.argmax(scores)])
            hist.append(y[obs].max())
        all_curves.append(hist)
    return np.array(all_curves)

c_oh = run_bo(X_oh, y_yield)
c_fp = run_bo(X_fp_pca, y_yield)
print(f"25 次实验后平均最高产率: One-Hot = {c_oh[:, -1].mean():.1f}% | Morgan FP + PCA = {c_fp[:, -1].mean():.1f}%")

**结果解析**：
- 引入分子结构指纹（Morgan FP + PCA）后，即使某个具体的添加剂或底物还没有被实验测过，模型也能根据它与已测分子的子结构相似性来推断其产率潜力，从而在早期更快锁定高产率区域；
- 同时可以看到，在这个全组合网格数据集中，连最简单的 One-Hot 编码配合 GPR 也能在 25 次实验内达到 93%+ 的极高产率——因为只要测过某个配体在几个反应中的表现，One-Hot 特征就能通过线性加和效应迅速识别出“哪些配体/碱本身是高活性组分”。